# NutraWise AI: Your Evidence-Based Nutraceutical Assistant

Welcome to NutraWise AI! This project is an advanced, conversational AI designed to provide personalized and evidence-based advice on nutraceuticals and supplements. It integrates Google's powerful Gemini model with a real-time PubMed scientific literature search to ensure recommendations are backed by data.

**Core Features:**
- **Personalized Profiles:** Creates detailed user health profiles to tailor recommendations.
- **Evidence-Based:** Searches the PubMed database to back suggestions with scientific studies.
- **Safety First:** Checks for potential interactions between supplements, medications, and health conditions.
- **Intelligent Conversation:** Remembers the context of your chat to provide relevant follow-up advice.
- **Comprehensive Database:** Comes pre-loaded with information on common supplements.

### 1. Initial Setup: Installing Dependencies

First things first, we need to install the necessary Python libraries. This cell uses `pip` to install Google's Generative AI SDK for interacting with the Gemini model and `requests` for making HTTP requests to the PubMed API.

In [1]:
# Install required packages (it's best to run this cell first in a new environment like Google Colab).
# The '!' character allows us to run shell commands directly from the notebook.
# We're installing 'google-generativeai' for the AI model and 'requests' for API calls.
!pip install google-generativeai requests

### 2. Importing Essential Libraries

Here, we import all the tools and modules we'll need throughout the project. This includes libraries for the AI, database management, data handling, and more.

In [2]:
# Core AI and API Libraries
import google.generativeai as genai # The main library for interacting with the Gemini AI model.
import requests                     # Used to make HTTP requests to the PubMed API for scientific articles.

# Data Handling and Structuring
import json                               # For working with JSON data formats.
import pandas as pd                       # A powerful library for data analysis (though not heavily used here, it's good practice to have).
from dataclasses import dataclass, asdict # Helps create simple, clean classes for storing data, like our user profile.
from typing import Dict, List, Optional   # Provides type hints for cleaner, more readable code.

# Database and System Libraries
import sqlite3                     # For creating and managing our local supplement database.
import os                          # Interacts with the operating system, useful for managing files and environment variables.
import logging                     # For logging events and errors, which helps in debugging.
import re                          # Stands for Regular Expressions, used for advanced string pattern matching.
from datetime import datetime      # To work with dates and times.

### 3. Securely Loading API Keys

To use services like Google AI and PubMed, we need API keys. To keep them secure, we load them from Google Colab's 'Secrets' manager. This prevents hardcoding sensitive keys directly in the code. If not running in Colab, it gracefully handles their absence.

In [3]:
# This block securely retrieves API keys when running in Google Colab.
# Using Colab's `userdata` is the recommended way to handle sensitive information.
try:
    # Attempt to import the Colab-specific userdata module.
    from google.colab import userdata

    # Fetch the keys from the Colab Secrets panel.
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    PUBMED_API_KEY = userdata.get('PubMed_API_KEY') # This one is optional but recommended for better API rates.
    print("API keys loaded successfully from Colab Secrets.")

except ImportError:
    # This error occurs if the code is not run in a Google Colab environment.
    print("Not running in Colab. API keys will need to be set manually as environment variables.")
    GOOGLE_API_KEY = None
    PUBMED_API_KEY = None

except Exception as e:
    # A general catch-all for any other errors during key loading.
    print(f"Error loading API keys: {str(e)}")
    GOOGLE_API_KEY = None
    PUBMED_API_KEY = None

API keys loaded successfully from Colab Secrets.


### 4. Setting Up Logging

Logging is crucial for understanding what the application is doing behind the scenes and for diagnosing any problems that might arise. We'll set up a basic logger to print informational messages and errors.

In [4]:
# Configure the logging system to display messages of level INFO and above.
# This helps in monitoring the application's flow and debugging issues.
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

### 5. Defining Core Data Structures

We use a `dataclass` to create a clean and simple structure for holding the user's health information. This acts like a blueprint for a user's profile.

In [5]:
@dataclass
class UserHealthProfile:
    """A structured class to hold all health-related information for a user.

    Using a dataclass automatically generates helpful methods like __init__ and __repr__,
    making it a clean and efficient way to store structured data.
    """
    age: int
    gender: str
    health_conditions: List[str]
    medications: List[str]
    goals: List[str]
    allergies: List[str]
    activity_level: str = "moderate" # Default value if not provided.

### 6. Building the User Interface Components

These classes manage the interactive command-line interface (CLI) that gathers information from the user. They are responsible for displaying menus and capturing user input in a structured way.

In [6]:
class InteractiveMenuManager:
    """A utility class to handle the creation and logic of interactive menus in the console.

    This keeps the user interface logic separate from the main application logic, which is good practice.
    """

    @staticmethod
    def display_single_choice_menu(title: str, options: List[str]) -> str:
        """Displays a menu where the user can only select one item.

        Args:
            title: The title to display for the menu.
            options: A list of strings representing the choices.

        Returns:
            The single option selected by the user.
        """
        print(f"\n{title}")
        print("-" * min(len(title), 40)) # Decorative line
        for i, option in enumerate(options, 1):
            print(f"{i}. {option}")

        # Loop until a valid choice is made.
        while True:
            try:
                choice = int(input(f"Choose (1-{len(options)}): "))
                if 1 <= choice <= len(options):
                    return options[choice - 1] # Return the selected string.
                else:
                    print(f"Invalid choice. Please enter a number between 1 and {len(options)}.")
            except ValueError:
                print("Invalid input. Please enter a number.")

    @staticmethod
    def display_multi_choice_menu(title: str, options: List[str]) -> List[str]:
        """Displays a menu where the user can select multiple items.

        Args:
            title: The title to display for the menu.
            options: A list of strings representing the choices.

        Returns:
            A list of all options selected by the user.
        """
        print(f"\n{title}")
        print("-" * min(len(title), 40))
        for i, option in enumerate(options, 1):
            print(f"{i}. {option}")
        print(f"{len(options) + 1}. Done") # Option to finish selection.

        selected = []

        # Loop to allow multiple selections.
        while True:
            try:
                choice = int(input(f"Choose (1-{len(options) + 1}): "))
                if choice == len(options) + 1:  # User is done.
                    break
                elif 1 <= choice <= len(options):
                    selected_option = options[choice - 1]
                    if selected_option not in selected:
                        selected.append(selected_option)
                        print(f"Added: {selected_option}")
                    else:
                        print(f"'{selected_option}' is already selected.")

                    # Special handling for the 'Other' option to allow custom input.
                    if selected_option == "Other":
                        custom_input = input("Please specify: ").strip()
                        if custom_input:
                            selected.remove("Other") # Remove the placeholder 'Other'.
                            selected.append(custom_input) # Add the user's custom value.
                            print(f"Added custom value: {custom_input}")
                else:
                    print(f"Invalid choice. Please enter a number between 1 and {len(options) + 1}.")
            except ValueError:
                print("Invalid input. Please enter a number.")

        return selected

In [7]:
class EnhancedProfileSetup:
    """Manages the entire user profile creation process using interactive menus.

    This class holds all the predefined options for health conditions, goals, etc.,
    and orchestrates the menu displays to build a complete UserHealthProfile object.
    """

    def __init__(self):
        """Initializes the profile setup with a menu manager and predefined choice lists."""
        self.menu_manager = InteractiveMenuManager()

        # --- Predefined options for menus to ensure consistency and ease of use ---
        self.gender_options = ["Male", "Female", "Other"]
        self.activity_options = ["Sedentary", "Light", "Moderate", "Active", "Very Active"]
        self.health_condition_options = [
            "Diabetes Type 1", "Diabetes Type 2", "Hypertension (High Blood Pressure)",
            "Heart Disease", "Asthma", "Arthritis", "High Cholesterol", "Depression/Anxiety",
            "Obesity", "Thyroid Disorders", "Osteoporosis", "Other"
        ]
        self.medication_options = [
            "Blood Pressure Medication", "Diabetes Medication (Metformin, Insulin)",
            "Cholesterol Medication (Statins)", "Pain Relievers (NSAIDs, Acetaminophen)",
            "Antidepressants", "Asthma Inhalers", "Blood Thinners (Warfarin, Aspirin)",
            "Thyroid Medication", "Birth Control Pills", "Vitamins/Supplements", "Other"
        ]
        self.health_goal_options = [
            "Weight Loss", "Weight Gain", "Muscle Building", "Improve Cardiovascular Health",
            "Manage Diabetes", "Lower Blood Pressure", "Reduce Stress & Anxiety",
            "Improve Sleep Quality", "Boost Energy Levels", "Strengthen Immune System", "Other"
        ]
        self.allergy_options = [
            "Peanuts", "Tree Nuts (Almonds, Walnuts, etc.)", "Shellfish", "Fish",
            "Milk/Dairy Products", "Eggs", "Soy Products", "Wheat/Gluten",
            "Sesame Seeds", "Food Dyes/Additives", "Other"
        ]

    def create_comprehensive_user_profile(self) -> UserHealthProfile:
        """Guides the user through a series of questions to create their health profile."""
        print("\n" + "="*50)
        print("HEALTH PROFILE SETUP")
        print("="*50)

        # --- Gather User Information Step-by-Step ---
        age = int(input("What is your age? "))
        gender = self.menu_manager.display_single_choice_menu("Select your gender:", self.gender_options)
        activity_level = self.menu_manager.display_single_choice_menu("Select your activity level:", self.activity_options).lower()
        health_conditions = self.menu_manager.display_multi_choice_menu("Any existing health conditions? (Select all that apply)", self.health_condition_options)
        medications = self.menu_manager.display_multi_choice_menu("Are you taking any medications?", self.medication_options)
        goals = self.menu_manager.display_multi_choice_menu("What are your primary health goals?", self.health_goal_options)
        allergies = self.menu_manager.display_multi_choice_menu("Do you have any allergies?", self.allergy_options)

        # --- Assemble the profile using the collected data ---
        profile = UserHealthProfile(
            age=age,
            gender=gender,
            health_conditions=health_conditions,
            medications=medications,
            goals=goals,
            allergies=allergies,
            activity_level=activity_level
        )

        print(f"\nProfile saved! ({age}-year-old {gender}, {activity_level} lifestyle)")
        return profile

    def _display_profile_summary(self, profile: UserHealthProfile):
        """Displays a neat summary of the user's current profile.
        The underscore `_` suggests this is an internal method, meant to be called from within the class or related functions.
        """
        print("\n" + "="*50)
        print("YOUR HEALTH PROFILE SUMMARY")
        print("="*50)
        print(f"Age & Lifestyle: {profile.age}-year-old {profile.gender}, {profile.activity_level.title()} lifestyle")

        if profile.health_conditions: print(f"Conditions: {', '.join(profile.health_conditions)}")
        if profile.medications: print(f"Medications: {', '.join(profile.medications)}")
        if profile.goals: print(f"Goals: {', '.join(profile.goals)}")
        if profile.allergies: print(f"Allergies: {', '.join(profile.allergies)}")

        print("="*50)

### 7. Integrating with Scientific Literature (PubMed)

This class is our gateway to scientific evidence. It connects to the NCBI PubMed API to search for research articles related to supplements and health conditions, adding a layer of credibility to the AI's suggestions.

In [8]:
class ScientificEvidenceSearcher:
    """Handles all interactions with the NCBI PubMed API to find scientific articles."""

    def __init__(self, api_key: str = None):
        """Initializes the searcher with the API base URLs and the user's API key."""
        self.base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
        self.search_url = self.base_url + "esearch.fcgi"  # Endpoint for searching
        self.summary_url = self.base_url + "esummary.fcgi" # Endpoint for getting article summaries
        self.api_key = api_key

    def search_scientific_articles(self, query: str, max_results: int = 3) -> List[Dict]:
        """Searches PubMed for articles matching a query and retrieves their details.

        Args:
            query: The search term (e.g., 'omega-3 and heart health').
            max_results: The maximum number of articles to return.

        Returns:
            A list of dictionaries, each containing details of a found article.
        """
        try:
            # Step 1: Search for article IDs (PMIDs) that match the query.
            search_params = {
                'db': 'pubmed',
                'term': query,
                'retmax': max_results,
                'retmode': 'json',
                'sort': 'relevance' # Sort results by relevance to the query.
            }
            if self.api_key: search_params['api_key'] = self.api_key # Use API key if available.

            search_response = requests.get(self.search_url, params=search_params, timeout=10)
            search_data = search_response.json()

            # If no results, return an empty list.
            if 'esearchresult' not in search_data or not search_data['esearchresult']['idlist']:
                return []

            # Step 2: Use the found IDs to fetch the summaries of the articles.
            pmids = search_data['esearchresult']['idlist']
            summary_params = {
                'db': 'pubmed',
                'id': ','.join(pmids), # Join IDs into a comma-separated string.
                'retmode': 'json'
            }
            if self.api_key: summary_params['api_key'] = self.api_key

            summary_response = requests.get(self.summary_url, params=summary_params, timeout=10)
            summary_data = summary_response.json()

            # Step 3: Format the retrieved data into a clean list of dictionaries.
            articles = []
            for pmid in pmids:
                if pmid in summary_data['result']:
                    article_data = summary_data['result'][pmid]
                    articles.append({
                        'pmid': pmid,
                        'title': article_data.get('title', 'No title available'),
                        'authors': self._extract_author_names(article_data.get('authors', [])),
                        'journal': article_data.get('source', 'Unknown journal'),
                        'pubdate': article_data.get('pubdate', 'Unknown date'),
                        'url': f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
                    })

            return articles

        except Exception as e:
            logger.error(f"An error occurred during PubMed search: {str(e)}")
            return [] # Return empty list on failure.

    def _extract_author_names(self, authors_data: List[Dict]) -> str:
        """A helper function to format the author list neatly (e.g., 'Author A, Author B, et al.')."""
        try:
            if not authors_data: return "Unknown authors"

            # Get the names of the first 3 authors.
            author_names = [author['name'] for author in authors_data[:3] if 'name' in author]

            # If there are more than 3, add 'et al.'
            if len(authors_data) > 3: author_names.append("et al.")

            return ", ".join(author_names) if author_names else "Unknown authors"
        except:
            return "Unknown authors" # Failsafe

### 8. The Core AI Engine: AdvancedNutriWiseAI

This is the heart of the application. The `AdvancedNutriWiseAI` class ties everything together:
- It initializes the Gemini AI model.
- It sets up and manages the local supplement database using SQLite.
- It uses the `ScientificEvidenceSearcher` to fetch research.
- It uses the `EnhancedProfileSetup` to create user profiles.
- It constructs complex prompts to send to the AI, combining user profile data, conversation history, and scientific evidence to get the most accurate and personalized response.

In [12]:
class AdvancedNutriWiseAI:
    """The main class that orchestrates the entire AI system."""

    def __init__(self, google_api_key: str, pubmed_api_key: str = None):
        """Initializes all components of the AI assistant."""
        # Configure the Gemini AI model with the provided API key.
        genai.configure(api_key=google_api_key)
        self.model = genai.GenerativeModel('gemini-2.5-flash') # Using the fast and efficient Flash model.

        # Initialize helper components.
        self.init_supplement_database()
        self.evidence_searcher = ScientificEvidenceSearcher(api_key=pubmed_api_key)
        self.profile_setup = EnhancedProfileSetup()

        # Keep track of the conversation to provide context.
        self.conversation_history = []

        print("AdvancedNutriWise AI initialized successfully!")
        print("Database loaded with supplement information.")
        if pubmed_api_key:
            print("Scientific evidence search is enabled with API Key.")
        else:
            print("Scientific evidence search is enabled (basic rate limits apply without API key).")

    def init_supplement_database(self):
        """Creates and populates a local SQLite database with supplement data."""
        # Connect to the database file (it will be created if it doesn't exist).
        self.conn = sqlite3.connect('advanced_nutriwise.db', check_same_thread=False)
        cursor = self.conn.cursor()

        # Define the structure (schema) of our supplements table.
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS supplements (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT UNIQUE, category TEXT, benefits TEXT, dosage_range TEXT,
                contraindications TEXT, interactions TEXT, side_effects TEXT,
                best_time TEXT, notes TEXT, vegan_friendly BOOLEAN, evidence_level TEXT
            )
        ''')

        # A list of tuples, where each tuple represents a row of supplement data.
        supplements_data = [
            ("Vitamin D3", "Vitamin", "Bone health, Immune support, Mood regulation", "1000-4000 IU daily", "Hypercalcemia, Kidney stones", "Warfarin, Thiazide diuretics", "Nausea at high doses", "Morning with fat", "Fat-soluble, check blood levels", True, "Strong"),
            ("Magnesium Glycinate", "Mineral", "Sleep quality, Muscle relaxation, Stress reduction", "200-400mg daily", "Severe kidney disease", "Antibiotics, Heart medications", "Diarrhea at high doses", "Evening", "Highly bioavailable form", True, "Strong"),
            ("Omega-3 (EPA/DHA)", "Fatty Acid", "Heart health, Brain function, Anti-inflammatory", "1000-2000mg daily", "Bleeding disorders", "Blood thinners", "Fishy aftertaste", "With meals", "Choose high EPA/DHA content", False, "Strong"),
            ("Probiotics", "Beneficial Bacteria", "Digestive health, Immune support", "10-50 billion CFU", "Severe immunocompromise", "Antibiotics reduce effectiveness", "Initial bloating, Gas", "Empty stomach", "Strain-specific benefits", True, "Moderate"),
            ("Ashwagandha", "Adaptogenic Herb", "Stress reduction, Cortisol regulation", "300-600mg daily", "Pregnancy, Autoimmune conditions", "Immunosuppressants, Sedatives", "Drowsiness, Stomach upset", "Evening", "Standardized to withanolides", True, "Moderate"),
            ("B-Complex", "Vitamin Complex", "Energy metabolism, Nervous system support", "1 capsule daily", "Rare, B6 toxicity at high doses", "Levodopa, Phenytoin", "Yellow urine (normal)", "Morning with food", "Water-soluble, excess excreted", True, "Strong"),
            ("Iron (Ferrous Bisglycinate)", "Mineral", "Anemia prevention, Energy", "18-65mg daily", "Hemochromatosis", "Calcium, Coffee, Tea", "Constipation, Dark stools", "Empty stomach with Vitamin C", "Gentle on the stomach", True, "Strong"),
            ("Zinc (Picolinate)", "Mineral", "Immune support, Wound healing", "8-15mg daily", "Wilson's disease", "Copper absorption, Antibiotics", "Nausea on empty stomach", "With food", "Don't exceed 40mg daily", True, "Strong"),
            ("Turmeric/Curcumin", "Anti-inflammatory Herb", "Anti-inflammatory, Joint health", "500-1000mg daily", "Gallstones, Bleeding disorders", "Blood thinners", "Stomach upset", "With meals and black pepper", "Piperine enhances absorption", True, "Moderate"),
            ("CoQ10 (Ubiquinol)", "Antioxidant", "Heart health, Energy production", "100-200mg daily", "None known", "Warfarin (monitor INR)", "Mild nausea", "With fats", "Ubiquinol form is more bioavailable", True, "Moderate")
        ]

        # Insert the data into the table. 'INSERT OR IGNORE' prevents errors if a supplement already exists.
        cursor.executemany('''
            INSERT OR IGNORE INTO supplements
            (name, category, benefits, dosage_range, contraindications, interactions,
            side_effects, best_time, notes, vegan_friendly, evidence_level)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', supplements_data)

        self.conn.commit() # Save the changes to the database.

    def create_user_profile(self) -> UserHealthProfile:
        """A convenience method to call the profile creation process."""
        return self.profile_setup.create_comprehensive_user_profile()

    def generate_ai_response(self, message: str, user_profile: UserHealthProfile = None) -> str:
        """The main AI response generation logic. This is where the magic happens.

        It builds a detailed prompt for the Gemini model, including the user's question,
        their health profile, the conversation history, and any relevant scientific evidence.
        """
        try:
            # A simple command to let the user see their profile.
            if any(keyword in message.lower() for keyword in ['my profile', 'show profile', 'my info']):
                if user_profile:
                    self.profile_setup._display_profile_summary(user_profile)
                    return "Here is your current health profile. You can ask to update it anytime."
                else:
                    return "No profile has been set up yet. Let's create one first!"

            # --- Prompt Engineering: Constructing the perfect prompt for the AI ---

            # 1. The System Message: Defines the AI's persona and rules.
            system_message = """You are NutraWish AI, a friendly and knowledgeable nutraceutical advisor.
            Your primary goal is to provide safe, evidence-based, and concise information.
            Always prioritize user safety, highlight potential interactions, and strongly recommend consulting a healthcare professional for medical advice.
            Use bullet points for lists and keep responses clear and easy to understand."""

            # 2. User Profile Context: Adds personalization.
            if user_profile:
                profile_context = f"USER PROFILE CONTEXT:\n{asdict(user_profile)}"
            else:
                profile_context = "No user profile is available. Provide general advice."

            # 3. Conversation History: Gives the AI memory.
            conversation_context = "\n".join([f"User: {turn['user']}\nAI: {turn['ai']}" for turn in self.conversation_history[-4:]])
            memory_context = f"RECENT CONVERSATION HISTORY:\n{conversation_context}"

            # 4. Scientific Evidence: Adds a layer of credibility.
            scientific_evidence = ""
            science_keywords = ['research', 'study', 'evidence', 'pubmed']
            if any(keyword in message.lower() for keyword in science_keywords):
                # A simple way to extract a topic from the user's message.
                search_query = message.replace("show me research on", "").strip()
                evidence_result = self.evidence_searcher.search_scientific_articles(search_query)
                if evidence_result:
                    scientific_evidence = "\n\nRECENT SCIENTIFIC EVIDENCE:\n"
                    for article in evidence_result:
                        scientific_evidence += f"- Title: {article['title']}\n  Link: {article['url']}\n"

            # --- Combine all parts into the final prompt ---
            full_prompt = f"""{system_message}

{profile_context}

{memory_context}

{scientific_evidence}

CURRENT USER MESSAGE: {message}"""

            # --- Send to Gemini and get the response ---
            response = self.model.generate_content(full_prompt)
            ai_response = response.text.strip()

            # Update our conversation history.
            self.conversation_history.append({'user': message, 'ai': ai_response})

            return ai_response

        except Exception as e:
            logger.error(f"Error generating AI response: {str(e)}")
            return f"I'm sorry, I encountered an error: {str(e)}. Please try again."

### 9. Launching the AI Chatbot

This is the main function that starts the interactive chat session. It greets the user, handles the initial profile setup, and then enters a loop to continuously listen for user input and provide AI-generated responses.

In [10]:
def launch_advanced_nutriwise_interface():
    """The main entry point for the user-facing chat interface."""
    print("\n" + "="*80)
    print("      WELCOME TO ADVANCED NUTRIWISE AI - YOUR NUTRACEUTICAL ASSISTANT")
    print("="*80)

    # --- Pre-launch Checks ---
    if not GOOGLE_API_KEY:
        print("\nFATAL ERROR: Google API key not found!")
        print("Please add 'GOOGLE_API_KEY' to your Colab Secrets (View > Secrets). The application cannot run without it.")
        return

    # --- Initialization ---
    try:
        ai = AdvancedNutriWiseAI(google_api_key=GOOGLE_API_KEY, pubmed_api_key=PUBMED_API_KEY)
    except Exception as e:
        print(f"\nFailed to initialize AI: {str(e)}")
        print("Please check your API keys and try again.")
        return

    # --- Onboarding and Profile Setup ---
    user_profile = None
    if input("\nWould you like to set up a health profile for personalized advice? (y/n): ").lower() == 'y':
        user_profile = ai.create_user_profile()

    # --- Welcome Message and Instructions ---
    print("\n" + "="*80)
    print("I'm ready to help! You can ask me things like:")
    print("  - 'What supplements can help with sleep?'")
    print("  - 'Check interactions between ashwagandha and my medications.'")
    print("  - 'Show me research on Vitamin D for immune support.'")
    print("\nType 'profile' to view/update your profile, or 'quit' to exit.")
    print("\nDISCLAIMER: I am an AI assistant. Always consult a healthcare professional for medical advice.")
    print("="*80)

    # --- Main Chat Loop ---
    chat_count = 0
    while True:
        try:
            user_input = input(f"\n[You]: ").strip()

            # --- Handle Commands ---
            if user_input.lower() in ['quit', 'exit', 'bye']:
                print("\nThank you for using NutraWish AI. Stay healthy!")
                break

            if user_input.lower() == 'profile':
                if user_profile:
                    ai.profile_setup._display_profile_summary(user_profile)
                    if input("\nUpdate profile? (y/n): ").lower() == 'y':
                        user_profile = ai.create_user_profile()
                else:
                    print("No profile found. Let's create one.")
                    user_profile = ai.create_user_profile()
                continue

            if not user_input: continue # Ignore empty input

            # --- Generate and Display AI Response ---
            print("\n[NutraWish AI]: Thinking...")
            response = ai.generate_ai_response(user_input, user_profile)
            print(f"\r[NutraWish AI]: {response}") # \r clears the 'Thinking...' message

            chat_count += 1
            print("-"*80)

        except KeyboardInterrupt:
            print("\n\nExiting session. Goodbye!")
            break
        except Exception as e:
            print(f"\nAn unexpected error occurred: {str(e)}")
            print("Please try rephrasing your question.")

### 10. Running the Application

The `if __name__ == "__main__":` block is a standard Python convention. It ensures that the `launch_advanced_nutriwise_interface()` function is called only when the script is executed directly (not when it's imported as a module into another script).

**To start the chat, run the cell below!**

In [13]:
# This is the standard entry point for a Python script.
# The code inside this block will only run when you execute this cell directly.
if __name__ == "__main__":
    launch_advanced_nutriwise_interface()


      WELCOME TO ADVANCED NUTRIWISE AI - YOUR NUTRACEUTICAL ASSISTANT
AdvancedNutriWise AI initialized successfully!
Database loaded with supplement information.
Scientific evidence search is enabled with API Key.

Would you like to set up a health profile for personalized advice? (y/n): n

I'm ready to help! You can ask me things like:
  - 'What supplements can help with sleep?'
  - 'Check interactions between ashwagandha and my medications.'
  - 'Show me research on Vitamin D for immune support.'

Type 'profile' to view/update your profile, or 'quit' to exit.

DISCLAIMER: I am an AI assistant. Always consult a healthcare professional for medical advice.

[You]: Check interactions between ashwagandha and shilajit

[NutraWish AI]: Thinking...
[NutraWish AI]: Hello there! As NutraWish AI, I'm here to provide you with safe and evidence-based information.

When considering interactions between ashwagandha and shilajit, here's what you should know:

*   **Generally Combined:** Both ashwag